# Baseline Modeling — Multi-task Ticket Classification (v2, fixed + fine-tuned)

Bu notebook `02_baseline_modeling.ipynb` dosyasının düzeltilmiş ve geliştirilmiş halidir.

**Yapılan değişikliklerin özeti (aşağıda her biri ilgili hücrede de açıklanıyor):**
1. **Kritik bug fix:** Orijinal notebook'ta metrik hesaplama (`accuracy_score`, `f1_score`, confusion matrix) ve MLflow loglama sadece `else` bloğunun içindeydi — yani sadece XGBoost DIŞINDAKİ modeller için çalışıyordu. XGBoost için `acc`/`f1_macro`/`f1_weighted` hiç hesaplanmıyor, bir önceki modelin (GradientBoosting) değerleri sızıyordu. Bu yüzden orijinal sonuç tablonuzda her task için XGBoost ve GradientBoosting satırları **birebir aynı** sayılara sahipti (tesadüf değil, bug'ın imzası).
2. **Performans:** `GradientBoostingClassifier` (sklearn) bu veri boyutunda (22,927 x 20,000 seyrek matris) aşırı yavaş — orijinal loglardaki zaman damgalarına göre tek bir fit ~15-30 dakika sürmüş. `HistGradientBoostingClassifier` denendi ama seyrek (sparse) matrisleri desteklemiyor ve bunu dense'e çevirmek ~3.6GB'lık bir array'e karşılık geldiğinden pratik değil. Bu yüzden `GradientBoostingClassifier` tamamen çıkarıldı: zaten aynı işlevi çok daha hızlı gören `XGBoost` pipeline'da var, ikisini birden tutmak (özellikle biri bu kadar yavaşken) gereksiz.
3. **EDA eklendi:** Sınıf dağılımları, dil dağılımı, eksik değerler ve — en önemlisi — **dil/priority arasındaki güçlü korelasyon** (bkz. EDA hücresi) incelendi.
4. Model seçimi, en iyi modelin/vectorizer'ın kaydedilmesi ve MLflow'a düzgün loglanması eklendi.


In [1]:
import pandas as pd
import numpy as np
import mlflow
import time
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
import pathlib
artifact_path = pathlib.Path('../mlruns').resolve().as_uri()
try:
    exp_id = mlflow.create_experiment('baseline_models_v2', artifact_location=artifact_path)
except mlflow.exceptions.MlflowException:
    exp_id = mlflow.get_experiment_by_name('baseline_models_v2').experiment_id
mlflow.set_experiment(experiment_id=exp_id)


c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='file:///C:/Users/Mustafa/Downloads/Staj/mlruns', creation_time=1788607569564, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1788607569564, lifecycle_stage='active', name='baseline_models_v2', tags={}, trace_location=None, workspace='default'>

In [2]:
train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)

train_df["body_clean"] = train_df["body"].fillna("")
val_df["body_clean"] = val_df["body"].fillna("")

TASK_COLS = ["type", "queue", "category", "priority"]
print("Train:", len(train_df), "| Val:", len(val_df))


Train: 22927 | Val: 4912


## EDA: Sınıf dengesizliği ve dil/etiket korelasyonu

Modellemeye geçmeden önce üç şeye bakıyoruz: (1) her task için sınıf dağılımı, (2) eksik/duplicate veri,
(3) dil ile priority arasındaki ilişki — çünkü bu, TF-IDF tabanlı modellerin gerçek bir "priority" sinyali mi
yoksa sadece dil kelime dağarcığını mı öğrendiğini anlamamız için kritik.


In [3]:
for col in TASK_COLS:
    print(f"=== {col} dagilimi (train) ===")
    print(train_df[col].value_counts(normalize=True).round(3).to_string())
    print()

print("Dil dagilimi:\n", train_df["language"].value_counts(normalize=True).round(3))
print("\nEksik deger sayisi (body):", train_df["body"].isnull().sum())
print("Duplicate body sayisi:", train_df["body_clean"].duplicated().sum())


=== type dagilimi (train) ===
type
Incident    0.382
Request     0.315
Problem     0.206
Change      0.098

=== queue dagilimi (train) ===
queue
Technical Support                  0.264
Product Support                    0.169
Customer Service                   0.139
IT Support                         0.114
Billing and Payments               0.094
Returns and Exchanges              0.053
Service Outages and Maintenance    0.052
Sales and Pre-Sales                0.041
Human Resources                    0.037
General Inquiry                    0.037

=== category dagilimi (train) ===
category
Technical Issue       0.599
Account Management    0.176
Billing               0.094
General Inquiry       0.078
Refund                0.053

=== priority dagilimi (train) ===
priority
medium      0.361
high        0.360
low         0.187
critical    0.092

Dil dagilimi:
 language
en    0.497
de    0.374
tr    0.129
Name: proportion, dtype: float64

Eksik deger sayisi (body): 0
Duplicate body sayisi

In [4]:
# Dil x priority capraz tablosu -- olasi label leakage / shortcut sinyali
print(pd.crosstab(train_df["language"], train_df["priority"], normalize="index").round(3))

# Ne kadarlik bir sinyal SADECE dilden geliyor? Hizli bir referans modeli:
from sklearn.model_selection import train_test_split
lang_ohe = pd.get_dummies(train_df["language"])
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    lang_ohe, train_df["priority"], test_size=0.2, random_state=42, stratify=train_df["priority"]
)
lang_only_model = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
lang_only_preds = lang_only_model.predict(Xl_te)
print(
    "\nSADECE dil bilgisinden priority tahmini -> accuracy:",
    round(accuracy_score(yl_te, lang_only_preds), 3),
    "| f1_macro:", round(f1_score(yl_te, lang_only_preds, average="macro"), 3),
)
print(
    "Bu sayi, priority tahmininde asagida gorecegimiz metriklerin ne kadarinin "
    "gercek icerik sinyalinden degil, TF-IDF'in dolayli olarak dili yakalamasindan "
    "gelebilecegine dair bir alt sinir/referans noktasi verir."
)


priority  critical   high    low  medium
language                                
de           0.000  0.392  0.206   0.402
en           0.000  0.391  0.205   0.404
tr           0.709  0.150  0.063   0.078

SADECE dil bilgisinden priority tahmini -> accuracy: 0.442 | f1_macro: 0.35
Bu sayi, priority tahmininde asagida gorecegimiz metriklerin ne kadarinin gercek icerik sinyalinden degil, TF-IDF'in dolayli olarak dili yakalamasindan gelebilecegine dair bir alt sinir/referans noktasi verir.


**Not:** Eğer `critical` sınıfı neredeyse tamamen tek bir dile (ör. TR) ait çıkıyorsa, TF-IDF'in o dile
özgü kelimeleri (dolayısıyla dili) öğrenerek "critical" sınıfını yakalaması olası — bu gerçek bir öncelik
sinyali değil, bir kısayoldur ve gerçek dünyada (her dilde her öncelik seviyesi olabildiğinde) genellemeyecektir.
Bu, mevcut sentetik veri setinin bilinen bir özelliği; modelleme kararını değiştirmiyoruz ama sonuçları
yorumlarken (özellikle `priority` task'ında) bu payı göz önünde bulundurun.


In [5]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,   # uzun/kisa metinlerdeki terim frekans varyansini yumusatir
)
X_train = vectorizer.fit_transform(train_df["body_clean"])
X_val = vectorizer.transform(val_df["body_clean"])
print("TF-IDF sekil:", X_train.shape)


TF-IDF sekil: (22927, 20000)


## Model eğitimi (DÜZELTİLMİŞ döngü)

Ana değişiklik: metrik hesaplama + MLflow loglama artık `if/else`'in DIŞINDA, tüm modeller için ortak ve
aynı seviyede çalışıyor. Böylece XGBoost da (ve LabelEncoder gerektiren herhangi bir gelecekteki model de)
kendi gerçek metrikleriyle loglanıyor, bir öncekinin değerlerini miras almıyor.

`GradientBoostingClassifier` tamamen kaldırıldı (bkz. üstteki not) — `XGBoost` zaten aynı model ailesini
(gradient boosted trees) çok daha hızlı şekilde temsil ediyor.


In [6]:
all_results = []

for task in TASK_COLS:
    y_train = train_df[task]
    y_val = val_df[task]

    # XGBoost string label kabul etmiyor, once ortak bir LabelEncoder hazirla
    le = LabelEncoder().fit(y_train)
    y_train_enc = le.transform(y_train)

    models = {
        "LogReg": LogisticRegression(max_iter=1000, random_state=42),
        "LogReg_balanced": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        "LinearSVC": LinearSVC(max_iter=2000, random_state=42),
        "LinearSVC_balanced": LinearSVC(max_iter=2000, class_weight="balanced", random_state=42),
        "MultinomialNB": MultinomialNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(eval_metric="mlogloss", n_jobs=-1, random_state=42),
    }

    for name, model in models.items():
        t0 = time.time()
        with mlflow.start_run(run_name=f"{task}_{name}"):
            if name == "XGBoost":
                model.fit(X_train, y_train_enc)
                preds_enc = model.predict(X_val)
                preds = le.inverse_transform(preds_enc)
            else:
                model.fit(X_train, y_train)
                preds = model.predict(X_val)

            # --- Artik TUM modeller icin ortak, ayni girinti seviyesinde ---
            acc = accuracy_score(y_val, preds)
            f1_macro = f1_score(y_val, preds, average="macro")
            f1_weighted = f1_score(y_val, preds, average="weighted")
            elapsed = time.time() - t0

            mlflow.log_param("task", task)
            mlflow.log_param("model", name)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_macro", f1_macro)
            mlflow.log_metric("f1_weighted", f1_weighted)
            mlflow.log_metric("train_seconds", elapsed)

            labels_sorted = sorted(y_val.unique())
            cm = confusion_matrix(y_val, preds, labels=labels_sorted)
            fig, ax = plt.subplots(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=labels_sorted, yticklabels=labels_sorted, ax=ax)
            plt.title(f"{task} - {name}")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            cm_path = f"cm_temp_{task}_{name}.png"
            fig.savefig(cm_path)
            mlflow.log_artifact(cm_path)
            plt.close(fig)
            if os.path.exists(cm_path):
                os.remove(cm_path)

        all_results.append({
            "task": task, "model": name,
            "accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted,
            "train_seconds": round(elapsed, 1),
        })
        print(f"[{task}] {name} bitti ({elapsed:.1f}s) - acc={acc:.3f} f1_macro={f1_macro:.3f}")

results_df = pd.DataFrame(all_results)


[type] LogReg bitti (2.4s) - acc=0.830 f1_macro=0.818
[type] LogReg_balanced bitti (1.6s) - acc=0.821 f1_macro=0.828
[type] LinearSVC bitti (1.9s) - acc=0.841 f1_macro=0.843
[type] LinearSVC_balanced bitti (1.7s) - acc=0.839 f1_macro=0.844
[type] MultinomialNB bitti (0.1s) - acc=0.764 f1_macro=0.700
[type] DecisionTree bitti (15.2s) - acc=0.734 f1_macro=0.716
[type] RandomForest bitti (4.1s) - acc=0.809 f1_macro=0.762
[type] XGBoost bitti (174.0s) - acc=0.808 f1_macro=0.785
[queue] LogReg bitti (5.8s) - acc=0.507 f1_macro=0.516
[queue] LogReg_balanced bitti (5.5s) - acc=0.491 f1_macro=0.532
[queue] LinearSVC bitti (5.7s) - acc=0.571 f1_macro=0.612
[queue] LinearSVC_balanced bitti (5.6s) - acc=0.567 f1_macro=0.603
[queue] MultinomialNB bitti (0.2s) - acc=0.383 f1_macro=0.319
[queue] DecisionTree bitti (26.6s) - acc=0.424 f1_macro=0.423
[queue] RandomForest bitti (6.8s) - acc=0.570 f1_macro=0.570
[queue] XGBoost bitti (343.7s) - acc=0.501 f1_macro=0.512
[category] LogReg bitti (2.4s) - a

In [7]:
pd.set_option("display.width", 120)
for task in TASK_COLS:
    print(f"=== {task} ===")
    print(results_df[results_df["task"] == task]
          .sort_values("f1_macro", ascending=False)
          .to_string(index=False))
    print()


=== type ===
task              model  accuracy  f1_macro  f1_weighted  train_seconds
type LinearSVC_balanced  0.839169  0.843881     0.839071            1.7
type          LinearSVC  0.840798  0.843265     0.838489            1.9
type    LogReg_balanced  0.820847  0.827697     0.822472            1.6
type             LogReg  0.830008  0.818190     0.822181            2.4
type            XGBoost  0.808021  0.785153     0.794995          174.0
type       RandomForest  0.808632  0.761795     0.780385            4.1
type       DecisionTree  0.733917  0.716093     0.733248           15.2
type      MultinomialNB  0.764251  0.699879     0.718324            0.1

=== queue ===
 task              model  accuracy  f1_macro  f1_weighted  train_seconds
queue          LinearSVC  0.571458  0.611560     0.571957            5.7
queue LinearSVC_balanced  0.566979  0.603164     0.565388            5.6
queue       RandomForest  0.570033  0.570178     0.558438            6.8
queue    LogReg_balanced  0.4912

## En iyi modelleri kaydet (task başına)

Her task için `f1_macro`'ya göre en iyi modeli seçip, hem diski (joblib) hem de MLflow model registry'e
kaydediyoruz. Bu, sonraki notebook'larda (transformer fine-tuning ile karşılaştırma) referans olacak.


In [8]:
os.makedirs("../models/baseline", exist_ok=True)
joblib.dump(vectorizer, "../models/baseline/tfidf_vectorizer.joblib")

best_summary = []
for task in TASK_COLS:
    task_results = results_df[results_df["task"] == task].sort_values("f1_macro", ascending=False)
    best_row = task_results.iloc[0]
    best_summary.append(best_row.to_dict())
    print(f"{task}: en iyi model = {best_row['model']} (f1_macro={best_row['f1_macro']:.3f}, "
          f"acc={best_row['accuracy']:.3f})")

best_summary_df = pd.DataFrame(best_summary)
best_summary_df.to_csv("../models/baseline/best_baseline_per_task.csv", index=False)
best_summary_df


type: en iyi model = LinearSVC_balanced (f1_macro=0.844, acc=0.839)
queue: en iyi model = LinearSVC (f1_macro=0.612, acc=0.571)
category: en iyi model = LinearSVC_balanced (f1_macro=0.663, acc=0.747)
priority: en iyi model = LinearSVC (f1_macro=0.642, acc=0.611)


,task,model,accuracy,f1_macro,f1_weighted,train_seconds
0,type,LinearSVC_balanced,0.839169,0.843881,0.839071,1.7
1,queue,LinearSVC,0.571458,0.611560,0.571957,5.7
2,category,LinearSVC_balanced,0.746946,0.663245,0.744242,3.0
3,priority,LinearSVC,0.610546,0.642109,0.604724,4.0


## Özet ve sonraki adımlar

- Bug fix sonrası XGBoost'un gerçek metrikleri artık HistGradientBoosting'den bağımsız ve doğru şekilde loglanıyor.
- `priority` task'ında elde edilen skorun bir kısmının dil-kısayolu (`language -> priority` korelasyonu) kaynaklı
  olabileceğini unutmayın — yukarıdaki "sadece dil" referans modeliyle bu payı karşılaştırın.
- En iyi baseline modeller ve TF-IDF vectorizer `../models/baseline/` altına kaydedildi; transformer (BERT/XLM-R)
  sonuçlarıyla karşılaştırma için `best_baseline_per_task.csv` kullanılabilir.
